# Iowa Corn Monthly Yield Retraining

This Colab notebook analyzes a merged CropNet monthly feature table and retrains a direct monthly-grain yield model.

Expected uploaded feature file:

`official_monthly_feature_table.parquet`

Target logic: each county-year-month feature row receives the USDA annual county yield label for that county-year. Training uses years `2017-2021`; testing uses `2022`.

In [ ]:
!pip -q install pandas numpy scikit-learn joblib matplotlib pyarrow


In [ ]:
from pathlib import Path
import re
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

RANDOM_STATE = 42
TRAIN_YEARS = list(range(2017, 2022))
TEST_YEAR = 2022

REFERENCE_PREVIOUS_YEAR_BASELINE = {
    'rmse': 24.02,
    'mae': 23.23,
    'r2': 0.231,
    'mape': 13.56,
}


## Upload and Load the Merged Monthly Feature Table

Upload the merged parquet file from your local project:

`outputs/experiments/corn_ia_monthly_2017_2022/artifacts/official_monthly_feature_table.parquet`

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    parquet_candidates = [name for name in uploaded if name.lower().endswith('.parquet')]
    if not parquet_candidates:
        raise ValueError('Please upload official_monthly_feature_table.parquet.')
    MONTHLY_PATH = Path(parquet_candidates[0])
except ImportError:
    MONTHLY_PATH = Path('official_monthly_feature_table.parquet')

print('Loading:', MONTHLY_PATH)
df = pd.read_parquet(MONTHLY_PATH)
print('Loaded shape:', df.shape)
df.head()


## Inspect the Dataset

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(list(df.columns))

print('\nDtypes:')
display(df.dtypes.rename('dtype').to_frame())

missing = df.isna().sum().sort_values(ascending=False)
print('\nMissing values:')
display(missing[missing > 0].to_frame('missing_count'))

if 'year' in df.columns:
    print('\nYears:', sorted(pd.to_numeric(df['year'], errors='coerce').dropna().astype(int).unique().tolist()))
if 'month' in df.columns:
    print('Months:', sorted(pd.to_numeric(df['month'], errors='coerce').dropna().astype(int).unique().tolist()))

county_col = 'county_id' if 'county_id' in df.columns else None
if county_col:
    print('County/FIPS count:', df[county_col].astype(str).nunique())

display(df.describe(include='all').T.head(80))


## Detect or Attach Yield Labels

If the uploaded parquet already includes a target column, this section uses it. Otherwise, upload USDA Corn county CSV files for 2017-2022 and the notebook will copy the annual county yield onto every matching monthly row.

In [ ]:
YIELD_COLUMN_CANDIDATES = [
    'yield_bu_acre',
    'YIELD, MEASURED IN BU / ACRE',
    'yield',
    'target_value',
]

def normalize_county_id(values):
    s = pd.Series(values).astype(str).str.extract(r'(\d+)', expand=False).fillna('')
    return s.str.zfill(5)

def normalize_crop_type(values):
    return pd.Series(values).astype(str).str.strip().str.lower().str.replace('_', ' ', regex=False).str.replace('-', ' ', regex=False)

def detect_yield_column(columns):
    for candidate in YIELD_COLUMN_CANDIDATES:
        if candidate in columns:
            return candidate
    for col in columns:
        lowered = str(col).lower()
        if 'yield' in lowered and ('acre' in lowered or lowered == 'yield'):
            return col
    return None

def infer_crop_from_filename(name):
    match = re.search(r'USDA_([^_]+(?:_[^_]+)*)_County_\d{4}\.csv$', Path(name).name)
    if match:
        return match.group(1).replace('_', ' ').lower()
    return 'corn'

def load_usda_label_csv(path):
    raw = pd.read_csv(path, dtype=str)
    yield_col = detect_yield_column(raw.columns)
    if yield_col is None:
        raise ValueError(f'Could not detect yield column in {path}. Columns: {list(raw.columns)}')

    lower_cols = {c.lower(): c for c in raw.columns}
    if 'year' in lower_cols:
        year = pd.to_numeric(raw[lower_cols['year']], errors='coerce')
    else:
        year_match = re.search(r'(20\d{2}|19\d{2})', Path(path).name)
        inferred_year = int(year_match.group(1)) if year_match else np.nan
        year = pd.Series([inferred_year] * len(raw))

    if 'state_ansi' in lower_cols and 'county_ansi' in lower_cols:
        state = raw[lower_cols['state_ansi']].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(2)
        county = raw[lower_cols['county_ansi']].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(3)
        county_id = state + county
    elif 'county_id' in lower_cols:
        county_id = normalize_county_id(raw[lower_cols['county_id']])
    elif 'fips' in lower_cols:
        county_id = normalize_county_id(raw[lower_cols['fips']])
    else:
        raise ValueError(f'Could not detect county FIPS columns in {path}.')

    crop_type = raw[lower_cols['commodity_desc']].str.lower() if 'commodity_desc' in lower_cols else infer_crop_from_filename(path)

    labels = pd.DataFrame({
        'county_id': county_id.astype(str).str.zfill(5),
        'year': pd.to_numeric(year, errors='coerce').astype('Int64'),
        'crop_type': normalize_crop_type(crop_type),
        'yield_bu_acre': pd.to_numeric(raw[yield_col], errors='coerce'),
        'target_unit': 'BU / ACRE',
    })
    return labels.dropna(subset=['county_id', 'year', 'yield_bu_acre'])

target_col = detect_yield_column(df.columns)
print('Detected target column:', target_col)

if 'county_id' in df.columns:
    df['county_id'] = normalize_county_id(df['county_id'])
if 'year' in df.columns:
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
if 'month' in df.columns:
    df['month'] = pd.to_numeric(df['month'], errors='coerce').astype('Int64')
if 'crop_type' in df.columns:
    df['crop_type'] = normalize_crop_type(df['crop_type'])
else:
    df['crop_type'] = 'corn'

if target_col is None:
    print('No yield target found in parquet. Upload USDA_Corn_County_2017.csv ... USDA_Corn_County_2022.csv now.')
    try:
        from google.colab import files
        usda_uploaded = files.upload()
        usda_paths = [name for name in usda_uploaded if name.lower().endswith('.csv')]
    except ImportError:
        usda_paths = sorted(str(p) for p in Path('.').glob('USDA_Corn_County_*.csv'))
    if not usda_paths:
        raise ValueError('No USDA CSV files were provided. Upload USDA Corn county yield CSVs and rerun this cell.')
    usda = pd.concat([load_usda_label_csv(path) for path in usda_paths], ignore_index=True)
    usda = usda[usda['crop_type'].eq('corn')].copy()
    print('USDA labels loaded:', usda.shape)
    display(usda.head())
    df = df.merge(usda, on=['county_id', 'year', 'crop_type'], how='inner')
    target_col = 'yield_bu_acre'
else:
    if target_col != 'yield_bu_acre':
        df['yield_bu_acre'] = pd.to_numeric(df[target_col], errors='coerce')
        target_col = 'yield_bu_acre'
    else:
        df[target_col] = pd.to_numeric(df[target_col], errors='coerce')

df = df.dropna(subset=[target_col, 'county_id', 'year', 'month']).copy()
print('Training frame shape after label handling:', df.shape)
print('Years:', sorted(df['year'].astype(int).unique().tolist()))
print('Months:', sorted(df['month'].astype(int).unique().tolist()))
print('County count:', df['county_id'].nunique())
print('\nYield summary:')
display(df[target_col].describe())
display(df.head())


## Feature Columns and Time Split

In [ ]:
df['month_sin'] = np.sin(2.0 * np.pi * df['month'].astype(float) / 12.0)
df['month_cos'] = np.cos(2.0 * np.pi * df['month'].astype(float) / 12.0)

identifier_patterns = [
    'county', 'fips', 'state', 'name', 'crop', 'commodity', 'unit', 'target', 'yield',
    'year', 'month', 'date', 'source', 'domain', 'period', 'desc', 'ansi', 'asd', 'agg_level',
]
explicit_exclude = {
    target_col, 'yield_bu_acre', 'target_unit', 'year', 'month', 'county_id', 'crop_type'
}

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = []
for col in numeric_cols:
    lowered = col.lower()
    if col in explicit_exclude:
        continue
    if any(pattern in lowered for pattern in identifier_patterns):
        if col not in {'month_sin', 'month_cos'}:
            continue
    feature_cols.append(col)

for timing_col in ['month_sin', 'month_cos']:
    if timing_col not in feature_cols:
        feature_cols.append(timing_col)

feature_cols = [col for col in feature_cols if col in df.columns]
print('Feature count:', len(feature_cols))
print(feature_cols)

train_df = df[df['year'].astype(int).isin(TRAIN_YEARS)].copy()
test_df = df[df['year'].astype(int).eq(TEST_YEAR)].copy()

if train_df.empty or test_df.empty:
    raise ValueError(f'Invalid split. Train rows={len(train_df)}, test rows={len(test_df)}. Check years in the dataset.')

print('Train rows:', len(train_df), 'Years:', sorted(train_df['year'].astype(int).unique().tolist()))
print('Test rows:', len(test_df), 'Years:', sorted(test_df['year'].astype(int).unique().tolist()))
print('Train counties:', train_df['county_id'].nunique(), 'Test counties:', test_df['county_id'].nunique())


## Train Baselines and Models

In [ ]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)

def metric_row(model, model_type, y_true, y_pred):
    return {
        'model': model,
        'model_type': model_type,
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'n_train': int(len(train_df)),
        'n_test': int(len(test_df)),
    }

X_train = train_df[feature_cols]
y_train = train_df[target_col].astype(float).to_numpy()
X_test = test_df[feature_cols]
y_test = test_df[target_col].astype(float).to_numpy()

predictions = test_df[['county_id', 'crop_type', 'year', 'month', target_col]].copy()
metrics = []
fitted_models = {}

# Baseline: train mean
train_mean = float(np.nanmean(y_train))
pred_train_mean = np.full(len(test_df), train_mean)
predictions['BaselineTrainMean'] = pred_train_mean
metrics.append(metric_row('BaselineTrainMean', 'baseline', y_test, pred_train_mean))

# Baseline: previous-year same-county yield
previous_lookup = (
    train_df[['county_id', 'year', target_col]]
    .drop_duplicates(['county_id', 'year'])
    .assign(year=lambda x: x['year'].astype(int))
    .set_index(['county_id', 'year'])[target_col]
    .to_dict()
)
pred_prev = np.array([
    previous_lookup.get((str(row.county_id).zfill(5), int(row.year) - 1), train_mean)
    for row in test_df[['county_id', 'year']].itertuples(index=False)
], dtype=float)
predictions['BaselinePreviousYearSameCounty'] = pred_prev
metrics.append(metric_row('BaselinePreviousYearSameCounty', 'baseline', y_test, pred_prev))

models = {
    'Ridge': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=10.0)),
    ]),
    'RandomForest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    'ExtraTrees': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', ExtraTreesRegressor(n_estimators=400, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    predictions[name] = pred
    metrics.append(metric_row(name, 'ml', y_test, pred))
    fitted_models[name] = pipe

metrics_df = pd.DataFrame(metrics).sort_values(['rmse', 'mae']).reset_index(drop=True)
metrics_df['rmse_delta_vs_reference_prev_year'] = metrics_df['rmse'] - REFERENCE_PREVIOUS_YEAR_BASELINE['rmse']
metrics_df['mae_delta_vs_reference_prev_year'] = metrics_df['mae'] - REFERENCE_PREVIOUS_YEAR_BASELINE['mae']
metrics_df['r2_delta_vs_reference_prev_year'] = metrics_df['r2'] - REFERENCE_PREVIOUS_YEAR_BASELINE['r2']
metrics_df['mape_delta_vs_reference_prev_year'] = metrics_df['mape'] - REFERENCE_PREVIOUS_YEAR_BASELINE['mape']
display(metrics_df)

best_ml_name = metrics_df[metrics_df['model_type'].eq('ml')].iloc[0]['model']
best_model = fitted_models[best_ml_name]
print('Best ML model:', best_ml_name)


## Save Artifacts

In [ ]:
predictions['best_model'] = best_ml_name
predictions['best_prediction'] = predictions[best_ml_name]
predictions['residual'] = predictions[target_col] - predictions['best_prediction']
predictions['abs_error'] = predictions['residual'].abs()
predictions['ape'] = np.where(predictions[target_col].abs() > 0, predictions['abs_error'] / predictions[target_col].abs() * 100.0, np.nan)

joblib.dump(best_model, 'best_yield_model.joblib')
metrics_df.to_csv('metrics.csv', index=False)
predictions.to_csv('predictions_2022.csv', index=False)
Path('feature_columns.txt').write_text('\n'.join(feature_cols), encoding='utf-8')

print('Saved:')
print('- best_yield_model.joblib')
print('- metrics.csv')
print('- predictions_2022.csv')
print('- feature_columns.txt')

try:
    from google.colab import files
    for artifact in ['best_yield_model.joblib', 'metrics.csv', 'predictions_2022.csv', 'feature_columns.txt']:
        files.download(artifact)
except ImportError:
    pass


## Charts

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(predictions[target_col], predictions['best_prediction'], alpha=0.65)
lo = min(predictions[target_col].min(), predictions['best_prediction'].min())
hi = max(predictions[target_col].max(), predictions['best_prediction'].max())
plt.plot([lo, hi], [lo, hi], linestyle='--')
plt.xlabel('Actual yield (bu/acre)')
plt.ylabel(f'Predicted yield ({best_ml_name})')
plt.title('Actual vs Predicted Yield, Test Year 2022')
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(predictions['residual'], bins=30, alpha=0.8)
plt.axvline(0, linestyle='--')
plt.xlabel('Residual: actual - predicted')
plt.ylabel('Count')
plt.title('Prediction Error Distribution')
plt.grid(alpha=0.25)
plt.show()


In [ ]:
tree_candidates = [name for name in ['ExtraTrees', 'RandomForest'] if name in fitted_models]
importance_model_name = best_ml_name if best_ml_name in tree_candidates else (tree_candidates[0] if tree_candidates else None)

if importance_model_name is not None:
    model_step = fitted_models[importance_model_name].named_steps['model']
    if hasattr(model_step, 'feature_importances_'):
        importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': model_step.feature_importances_,
        }).sort_values('importance', ascending=False)
        display(importance.head(30))
        plt.figure(figsize=(8, 8))
        top = importance.head(25).iloc[::-1]
        plt.barh(top['feature'], top['importance'])
        plt.xlabel('Importance')
        plt.title(f'Top Feature Importances ({importance_model_name})')
        plt.tight_layout()
        plt.show()
else:
    print('No tree model was available for feature importance.')


## Quick Interpretation

Use `metrics.csv` to compare the full-data monthly model against the known smoke baseline. A negative RMSE/MAE delta versus the reference baseline means the model improved over the old small-data previous-year baseline.